In [1]:
import pandas as pd

# look at D1NAMO insulin data for patient 001
insulin_df = pd.read_csv('data/raw/d1namo/diabetes_subset_pictures-glucose-food-insulin/001/insulin.csv')
print(insulin_df.shape)
print(insulin_df.dtypes)
print(insulin_df.head(20))

(19, 5)
date                str
time                str
fast_insulin      int64
slow_insulin    float64
comment         float64
dtype: object
          date      time  fast_insulin  slow_insulin  comment
0   2014-10-01  10:06:00             7           NaN      NaN
1   2014-10-01  16:50:00             4           NaN      NaN
2   2014-10-01  19:28:00             6           NaN      NaN
3   2014-10-01  22:27:00             8           NaN      NaN
4   2014-10-01  23:48:00             0          31.0      NaN
5   2014-10-02  00:45:00             2           NaN      NaN
6   2014-10-02  10:10:00             3           NaN      NaN
7   2014-10-02  12:34:00            12           NaN      NaN
8   2014-10-02  21:24:00             5           NaN      NaN
9   2014-10-02  22:24:00             4          31.0      NaN
10  2014-10-03  06:11:00             9           NaN      NaN
11  2014-10-03  09:30:00             3           NaN      NaN
12  2014-10-03  19:05:00             7           NaN

In [2]:
import pandas as pd
import numpy as np

def compute_iob_series(glucose_df, insulin_df, peak_mins=60, duration_mins=240):
    """
    For each glucose timestamp, compute total IOB from all active boluses
    using triangular decay. Only uses fast_insulin (bolus doses).
    """
    # parse timestamps
    insulin_df = insulin_df.copy()
    insulin_df['timestamp'] = pd.to_datetime(
        insulin_df['date'] + ' ' + insulin_df['time']
    )
    # only bolus doses, drop zeros
    boluses = insulin_df[insulin_df['fast_insulin'] > 0][
        ['timestamp', 'fast_insulin']
    ].copy()
    
    glucose_df = glucose_df.copy()
    iob_values = []
    
    for t in glucose_df['timestamp']:
        total_iob = 0.0
        for _, bolus in boluses.iterrows():
            mins_elapsed = (t - bolus['timestamp']).total_seconds() / 60
            if 0 <= mins_elapsed <= duration_mins:
                dose = bolus['fast_insulin']
                if mins_elapsed <= peak_mins:
                    # rising phase
                    activity = (mins_elapsed / peak_mins)
                else:
                    # falling phase
                    activity = (duration_mins - mins_elapsed) / (duration_mins - peak_mins)
                # remaining IOB = dose * fraction not yet absorbed
                remaining = max(0, 1 - (mins_elapsed / duration_mins))
                total_iob += dose * remaining
        iob_values.append(total_iob)
    
    glucose_df['iob'] = iob_values
    return glucose_df

# load patient 001 glucose to test
glucose_df = pd.read_csv(
    'data/raw/d1namo/diabetes_subset_pictures-glucose-food-insulin/001/glucose.csv'
)
print(glucose_df.columns.tolist())
print(glucose_df.head())

['date', 'time', 'glucose', 'type', 'comments']
         date      time  glucose    type  comments
0  2014-10-01  19:14:00     10.3     cgm       NaN
1  2014-10-01  19:19:00      9.9     cgm       NaN
2  2014-10-01  19:23:00      9.4  manual       NaN
3  2014-10-01  19:24:00      9.8     cgm       NaN
4  2014-10-01  19:29:00      9.6     cgm       NaN


In [3]:
# load and prep glucose for patient 001
glucose_df = glucose_df[glucose_df['type'] == 'cgm'].copy()
glucose_df['timestamp'] = pd.to_datetime(glucose_df['date'] + ' ' + glucose_df['time'])
glucose_df['glucose_mgdl'] = glucose_df['glucose'] * 18.018  # mmol/L → mg/dL
glucose_df = glucose_df[['timestamp', 'glucose_mgdl']].reset_index(drop=True)

# load insulin for patient 001
insulin_df = pd.read_csv(
    'data/raw/d1namo/diabetes_subset_pictures-glucose-food-insulin/001/insulin.csv'
)

# run IOB computation
glucose_with_iob = compute_iob_series(glucose_df, insulin_df)

# sanity check
print(glucose_with_iob[['timestamp', 'glucose_mgdl', 'iob']].head(20))
print("\nIOB stats:")
print(glucose_with_iob['iob'].describe())
print(f"\nNon-zero IOB readings: {(glucose_with_iob['iob'] > 0).sum()} / {len(glucose_with_iob)}")

             timestamp  glucose_mgdl       iob
0  2014-10-01 19:14:00      185.5854  1.600000
1  2014-10-01 19:19:00      178.3782  1.516667
2  2014-10-01 19:24:00      176.5764  1.433333
3  2014-10-01 19:29:00      172.9728  7.325000
4  2014-10-01 19:34:00      169.3692  7.116667
5  2014-10-01 19:39:00      165.7656  6.908333
6  2014-10-01 19:44:00      160.3602  6.700000
7  2014-10-01 19:49:00      156.7566  6.491667
8  2014-10-01 19:54:00      151.3512  6.283333
9  2014-10-01 19:59:00      147.7476  6.075000
10 2014-10-01 20:04:00      144.1440  5.866667
11 2014-10-01 20:09:00      142.3422  5.658333
12 2014-10-01 20:14:00      142.3422  5.450000
13 2014-10-01 20:19:00      140.5404  5.241667
14 2014-10-01 20:24:00      140.5404  5.033333
15 2014-10-01 20:29:00      138.7386  4.825000
16 2014-10-01 20:34:00      135.1350  4.616667
17 2014-10-01 20:39:00      135.1350  4.408333
18 2014-10-01 20:44:00      135.1350  4.200000
19 2014-10-01 20:49:00      136.9368  3.991667

IOB stats:
c

In [4]:
PATIENT_IDS = ['001', '002', '003', '004', '005', 
               '006', '007', '008', '009']
BASE_PATH = 'data/raw/d1namo/diabetes_subset_pictures-glucose-food-insulin'

all_results = []

for pid in PATIENT_IDS:
    try:
        g = pd.read_csv(f'{BASE_PATH}/{pid}/glucose.csv')
        i = pd.read_csv(f'{BASE_PATH}/{pid}/insulin.csv')
        
        g = g[g['type'] == 'cgm'].copy()
        g['timestamp'] = pd.to_datetime(g['date'] + ' ' + g['time'])
        g['glucose_mgdl'] = g['glucose'] * 18.018
        g = g[['timestamp', 'glucose_mgdl']].reset_index(drop=True)
        
        g_iob = compute_iob_series(g, i)
        g_iob['patient_id'] = pid
        all_results.append(g_iob)
        
        nonzero = (g_iob['iob'] > 0).sum()
        print(f"Patient {pid}: {len(g_iob)} readings, "
              f"IOB non-zero: {nonzero} ({100*nonzero/len(g_iob):.1f}%), "
              f"max IOB: {g_iob['iob'].max():.2f}")
        
    except Exception as e:
        print(f"Patient {pid}: ERROR — {e}")

combined_df = pd.concat(all_results, ignore_index=True)
print(f"\nTotal readings across all patients: {len(combined_df)}")
print(f"Overall IOB stats:\n{combined_df['iob'].describe()}")

Patient 001: 1413 readings, IOB non-zero: 669 (47.3%), max IOB: 13.20
Patient 002: 1056 readings, IOB non-zero: 480 (45.5%), max IOB: 7.97
Patient 003: 183 readings, IOB non-zero: 48 (26.2%), max IOB: 7.10
Patient 004: 969 readings, IOB non-zero: 395 (40.8%), max IOB: 9.83
Patient 005: 909 readings, IOB non-zero: 258 (28.4%), max IOB: 5.50
Patient 006: 1280 readings, IOB non-zero: 367 (28.7%), max IOB: 7.34
Patient 007: 988 readings, IOB non-zero: 551 (55.8%), max IOB: 8.00
Patient 008: 1140 readings, IOB non-zero: 656 (57.5%), max IOB: 9.79
Patient 009: 117 readings, IOB non-zero: 75 (64.1%), max IOB: 3.99

Total readings across all patients: 8055
Overall IOB stats:
count    8055.000000
mean        1.231675
std         2.026108
min         0.000000
25%         0.000000
50%         0.000000
75%         1.950347
max        13.200000
Name: iob, dtype: float64


In [5]:
from sklearn.preprocessing import StandardScaler

def engineer_features_with_iob(glucose_iob_df):
    """
    Extends your existing feature engineering to include IOB as feature 9.
    Input: dataframe with timestamp, glucose_mgdl, iob columns
    Output: dataframe with all 9 features ready for windowing
    """
    df = glucose_iob_df.copy().sort_values('timestamp').reset_index(drop=True)
    
    # existing features from your dataset.py
    df['glucose_norm'] = (df['glucose_mgdl'] - 40) / (400 - 40)
    df['delta_1'] = df['glucose_mgdl'].diff(1)
    df['delta_3'] = df['glucose_mgdl'].diff(3)
    df['delta_6'] = df['glucose_mgdl'].diff(6)
    df['rolling_mean_12'] = df['glucose_mgdl'].rolling(12).mean()
    df['rolling_std_12'] = df['glucose_mgdl'].rolling(12).std()
    df['hour_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.hour / 24)
    
    # new feature — z-score normalise IOB same as delta features
    iob_mean = df['iob'].mean()
    iob_std = df['iob'].std()
    df['iob_norm'] = (df['iob'] - iob_mean) / (iob_std + 1e-8)
    
    # drop NaNs from diff/rolling operations
    df = df.dropna().reset_index(drop=True)
    
    return df

# test on patient 001
patient_001 = all_results[0]
features_001 = engineer_features_with_iob(patient_001)

FEATURE_COLS = ['glucose_norm', 'delta_1', 'delta_3', 'delta_6',
                'rolling_mean_12', 'rolling_std_12', 
                'hour_sin', 'hour_cos', 'iob_norm']

print(f"Feature columns: {FEATURE_COLS}")
print(f"Shape after feature engineering: {features_001[FEATURE_COLS].shape}")
print(f"\nSample row:\n{features_001[FEATURE_COLS].iloc[0]}")
print(f"\nAny NaNs: {features_001[FEATURE_COLS].isna().any().any()}")
print(f"\nIOB norm stats:\n{features_001['iob_norm'].describe()}")

Feature columns: ['glucose_norm', 'delta_1', 'delta_3', 'delta_6', 'rolling_mean_12', 'rolling_std_12', 'hour_sin', 'hour_cos', 'iob_norm']
Shape after feature engineering: (1402, 9)

Sample row:
glucose_norm         0.284284
delta_1             -1.801800
delta_3             -9.009000
delta_6            -23.423400
rolling_mean_12    162.612450
rolling_std_12      14.375958
hour_sin            -0.866025
hour_cos             0.500000
iob_norm             1.322393
Name: 0, dtype: float64

Any NaNs: False

IOB norm stats:
count    1402.000000
mean       -0.009141
std         0.995985
min        -0.666141
25%        -0.666141
50%        -0.666141
75%         0.420207
max         3.972796
Name: iob_norm, dtype: float64


In [6]:
def create_windows(feature_df, feature_cols, window_size=24, horizon=6):
    """
    Sliding window construction.
    Returns X: (n, 24, 9), y: (n,)
    """
    data = feature_df[feature_cols].values
    target = feature_df['glucose_norm'].values
    
    X, y = [], []
    for i in range(len(data) - window_size - horizon):
        X.append(data[i:i + window_size])
        y.append(target[i + window_size + horizon])
    
    return np.array(X), np.array(y)

# build windows for all 9 patients with temporal 80/20 split
X_train_all, y_train_all = [], []
X_val_all, y_val_all = [], []

for idx, pid in enumerate(PATIENT_IDS):
    patient_df = all_results[idx]
    features_df = engineer_features_with_iob(patient_df)
    
    # temporal 80/20 split per patient
    split = int(len(features_df) * 0.8)
    train_df = features_df.iloc[:split]
    val_df = features_df.iloc[split:]
    
    X_tr, y_tr = create_windows(train_df, FEATURE_COLS)
    X_val, y_val = create_windows(val_df, FEATURE_COLS)
    
    X_train_all.append(X_tr)
    y_train_all.append(y_tr)
    X_val_all.append(X_val)
    y_val_all.append(y_val)
    
    print(f"Patient {pid}: train {X_tr.shape}, val {X_val.shape}")

X_train = np.concatenate(X_train_all)
y_train = np.concatenate(y_train_all)
X_val = np.concatenate(X_val_all)
y_val = np.concatenate(y_val_all)

print(f"\nFinal training set: {X_train.shape}")
print(f"Final validation set: {X_val.shape}")
print(f"Feature dimension: {X_train.shape[2]} (should be 9)")

Patient 001: train (1091, 24, 9), val (251, 24, 9)
Patient 002: train (806, 24, 9), val (179, 24, 9)
Patient 003: train (107, 24, 9), val (5, 24, 9)
Patient 004: train (736, 24, 9), val (162, 24, 9)
Patient 005: train (688, 24, 9), val (150, 24, 9)
Patient 006: train (985, 24, 9), val (224, 24, 9)
Patient 007: train (751, 24, 9), val (166, 24, 9)
Patient 008: train (873, 24, 9), val (196, 24, 9)
Patient 009: train (54, 24, 9), val (0,)


ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 3 dimension(s) and the array at index 8 has 1 dimension(s)

In [7]:
X_train_all, y_train_all = [], []
X_val_all, y_val_all = [], []

for idx, pid in enumerate(PATIENT_IDS):
    patient_df = all_results[idx]
    features_df = engineer_features_with_iob(patient_df)
    
    split = int(len(features_df) * 0.8)
    train_df = features_df.iloc[:split]
    val_df = features_df.iloc[split:]
    
    X_tr, y_tr = create_windows(train_df, FEATURE_COLS)
    X_val, y_val = create_windows(val_df, FEATURE_COLS)
    
    # only include if windows were actually created
    if X_tr.shape[0] > 0:
        X_train_all.append(X_tr)
        y_train_all.append(y_tr)
    
    if X_val.shape[0] > 0:
        X_val_all.append(X_val)
        y_val_all.append(y_val)
    
    print(f"Patient {pid}: train {X_tr.shape}, val {X_val.shape}")

X_train = np.concatenate(X_train_all)
y_train = np.concatenate(y_train_all)
X_val = np.concatenate(X_val_all)
y_val = np.concatenate(y_val_all)

print(f"\nFinal training set: {X_train.shape}")
print(f"Final validation set: {X_val.shape}")
print(f"Feature dimension: {X_train.shape[2]} (should be 9)")

Patient 001: train (1091, 24, 9), val (251, 24, 9)
Patient 002: train (806, 24, 9), val (179, 24, 9)
Patient 003: train (107, 24, 9), val (5, 24, 9)
Patient 004: train (736, 24, 9), val (162, 24, 9)
Patient 005: train (688, 24, 9), val (150, 24, 9)
Patient 006: train (985, 24, 9), val (224, 24, 9)
Patient 007: train (751, 24, 9), val (166, 24, 9)
Patient 008: train (873, 24, 9), val (196, 24, 9)
Patient 009: train (54, 24, 9), val (0,)

Final training set: (6091, 24, 9)
Final validation set: (1333, 24, 9)
Feature dimension: 9 (should be 9)


In [9]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# updated model with input_size=9
class GlucoseLSTM(nn.Module):
    def __init__(self, input_size=9, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze()

# tensors
X_tr = torch.FloatTensor(X_train)
y_tr = torch.FloatTensor(y_train)
X_v = torch.FloatTensor(X_val)
y_v = torch.FloatTensor(y_val)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), 
                         batch_size=32, shuffle=True)

# training setup
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Training on: {device}")

model = GlucoseLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)
# training loop
best_val_loss = float('inf')
best_epoch = 0

for epoch in range(50):
    # train
    model.train()
    train_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    
    # validate
    model.eval()
    with torch.no_grad():
        val_pred = model(X_v.to(device))
        val_loss = criterion(val_pred, y_v.to(device)).item()
    
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), 'models/glucose_lstm_iob.pt')
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train loss: {np.mean(train_losses):.4f} "
              f"| val loss: {val_loss:.4f}")

print(f"\nBest val loss: {best_val_loss:.4f} at epoch {best_epoch}")

Training on: mps
Epoch   0 | train loss: 0.0144 | val loss: 0.0252
Epoch  10 | train loss: 0.0043 | val loss: 0.0076
Epoch  20 | train loss: 0.0035 | val loss: 0.0090
Epoch  30 | train loss: 0.0027 | val loss: 0.0092
Epoch  40 | train loss: 0.0025 | val loss: 0.0089

Best val loss: 0.0070 at epoch 15


In [10]:
# load best checkpoint and evaluate
model.load_state_dict(torch.load('models/glucose_lstm_iob.pt', 
                                  map_location=device))
model.eval()

with torch.no_grad():
    val_pred = model(X_v.to(device)).cpu().numpy()

# denormalise predictions and targets back to mg/dL
GLUCOSE_MIN, GLUCOSE_MAX = 40, 400

y_val_mgdl = y_val * (GLUCOSE_MAX - GLUCOSE_MIN) + GLUCOSE_MIN
val_pred_mgdl = val_pred * (GLUCOSE_MAX - GLUCOSE_MIN) + GLUCOSE_MIN

# metrics
mae = np.mean(np.abs(val_pred_mgdl - y_val_mgdl))
rmse = np.sqrt(np.mean((val_pred_mgdl - y_val_mgdl) ** 2))

# hypoglycemia direction accuracy
HYPO_THRESHOLD = 70
actual_hypo = y_val_mgdl < HYPO_THRESHOLD
pred_hypo = val_pred_mgdl < HYPO_THRESHOLD
direction_acc = np.mean(actual_hypo == pred_hypo) * 100

print(f"=== DiabLoom v2 (with IOB) ===")
print(f"MAE:  {mae:.2f} mg/dL  (previous: 18.5)")
print(f"RMSE: {rmse:.2f} mg/dL")
print(f"Hypo direction accuracy: {direction_acc:.1f}%  (previous: 94.4%)")
print(f"\nSample predictions vs actuals (mg/dL):")
for i in range(10):
    print(f"  predicted: {val_pred_mgdl[i]:.1f} | actual: {y_val_mgdl[i]:.1f} | "
          f"diff: {abs(val_pred_mgdl[i]-y_val_mgdl[i]):.1f}")

=== DiabLoom v2 (with IOB) ===
MAE:  22.92 mg/dL  (previous: 18.5)
RMSE: 30.21 mg/dL
Hypo direction accuracy: 93.9%  (previous: 94.4%)

Sample predictions vs actuals (mg/dL):
  predicted: 206.4 | actual: 205.4 | diff: 1.0
  predicted: 177.3 | actual: 198.2 | diff: 20.9
  predicted: 178.5 | actual: 192.8 | diff: 14.3
  predicted: 185.5 | actual: 185.6 | diff: 0.1
  predicted: 159.6 | actual: 183.8 | diff: 24.1
  predicted: 167.9 | actual: 183.8 | diff: 15.9
  predicted: 151.0 | actual: 185.6 | diff: 34.5
  predicted: 142.6 | actual: 189.2 | diff: 46.5
  predicted: 139.2 | actual: 192.8 | diff: 53.6
  predicted: 148.3 | actual: 194.6 | diff: 46.3


In [11]:
def engineer_features_with_iob_v2(glucose_iob_df, iob_max=15.0):
    """
    Same as before but uses min-max for IOB instead of z-score.
    iob_max=15 covers realistic bolus range for T1D patients.
    """
    df = glucose_iob_df.copy().sort_values('timestamp').reset_index(drop=True)
    
    df['glucose_norm'] = (df['glucose_mgdl'] - 40) / (400 - 40)
    df['delta_1'] = df['glucose_mgdl'].diff(1)
    df['delta_3'] = df['glucose_mgdl'].diff(3)
    df['delta_6'] = df['glucose_mgdl'].diff(6)
    df['rolling_mean_12'] = df['glucose_mgdl'].rolling(12).mean()
    df['rolling_std_12'] = df['glucose_mgdl'].rolling(12).std()
    df['hour_sin'] = np.sin(2 * np.pi * df['timestamp'].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['timestamp'].dt.hour / 24)
    
    # min-max instead of z-score — 0 IOB maps to 0.0 cleanly
    df['iob_norm'] = np.clip(df['iob'] / iob_max, 0, 1)
    
    df = df.dropna().reset_index(drop=True)
    return df

# rebuild windows with v2 normalization
X_train_all, y_train_all = [], []
X_val_all, y_val_all = [], []

for idx, pid in enumerate(PATIENT_IDS):
    patient_df = all_results[idx]
    features_df = engineer_features_with_iob_v2(patient_df)
    
    split = int(len(features_df) * 0.8)
    train_df = features_df.iloc[:split]
    val_df = features_df.iloc[split:]
    
    X_tr, y_tr = create_windows(train_df, FEATURE_COLS)
    X_val, y_val = create_windows(val_df, FEATURE_COLS)
    
    if X_tr.shape[0] > 0:
        X_train_all.append(X_tr)
        y_train_all.append(y_tr)
    if X_val.shape[0] > 0:
        X_val_all.append(X_val)
        y_val_all.append(y_val)

X_train = np.concatenate(X_train_all)
y_train = np.concatenate(y_train_all)
X_val = np.concatenate(X_val_all)
y_val = np.concatenate(y_val_all)

# check IOB distribution with new normalization
all_iob = np.concatenate([
    engineer_features_with_iob_v2(all_results[i])['iob_norm'].values 
    for i in range(len(PATIENT_IDS))
])
print(f"IOB norm v2 stats:")
print(f"  min: {all_iob.min():.3f}, max: {all_iob.max():.3f}")
print(f"  mean: {all_iob.mean():.3f}, zeros: {(all_iob==0).mean()*100:.1f}%")
print(f"\nWindows: train {X_train.shape}, val {X_val.shape}")

IOB norm v2 stats:
  min: 0.000, max: 0.880
  mean: 0.081, zeros: 56.8%

Windows: train (6091, 24, 9), val (1333, 24, 9)


In [12]:
# retrain with v2 normalization
X_tr = torch.FloatTensor(X_train)
y_tr = torch.FloatTensor(y_train)
X_v = torch.FloatTensor(X_val)
y_v = torch.FloatTensor(y_val)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), 
                         batch_size=32, shuffle=True)

model_v2 = GlucoseLSTM().to(device)
optimizer = torch.optim.Adam(model_v2.parameters(), lr=1e-3)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5
)

best_val_loss = float('inf')
best_epoch = 0

for epoch in range(50):
    model_v2.train()
    train_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model_v2(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())
    
    model_v2.eval()
    with torch.no_grad():
        val_pred = model_v2(X_v.to(device))
        val_loss = criterion(val_pred, y_v.to(device)).item()
    
    scheduler.step(val_loss)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        torch.save(model_v2.state_dict(), 'models/glucose_lstm_iob_v2.pt')
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | train loss: {np.mean(train_losses):.4f} "
              f"| val loss: {val_loss:.4f}")

print(f"\nBest val loss: {best_val_loss:.4f} at epoch {best_epoch}")

# evaluate immediately
model_v2.load_state_dict(torch.load('models/glucose_lstm_iob_v2.pt',
                                     map_location=device))
model_v2.eval()
with torch.no_grad():
    val_pred = model_v2(X_v.to(device)).cpu().numpy()

y_val_mgdl = y_val * (GLUCOSE_MAX - GLUCOSE_MIN) + GLUCOSE_MIN
val_pred_mgdl = val_pred * (GLUCOSE_MAX - GLUCOSE_MIN) + GLUCOSE_MIN

mae = np.mean(np.abs(val_pred_mgdl - y_val_mgdl))
rmse = np.sqrt(np.mean((val_pred_mgdl - y_val_mgdl) ** 2))
actual_hypo = y_val_mgdl < HYPO_THRESHOLD
pred_hypo = val_pred_mgdl < HYPO_THRESHOLD
direction_acc = np.mean(actual_hypo == pred_hypo) * 100

print(f"\n=== DiabLoom v2 min-max IOB ===")
print(f"MAE:  {mae:.2f} mg/dL  (original: 18.5)")
print(f"RMSE: {rmse:.2f} mg/dL  (original: ~24)")
print(f"Hypo direction accuracy: {direction_acc:.1f}%  (original: 94.4%)")

Epoch   0 | train loss: 0.0185 | val loss: 0.0367
Epoch  10 | train loss: 0.0045 | val loss: 0.0105
Epoch  20 | train loss: 0.0035 | val loss: 0.0071
Epoch  30 | train loss: 0.0031 | val loss: 0.0076
Epoch  40 | train loss: 0.0025 | val loss: 0.0080

Best val loss: 0.0063 at epoch 29

=== DiabLoom v2 min-max IOB ===
MAE:  23.07 mg/dL  (original: 18.5)
RMSE: 28.51 mg/dL  (original: ~24)
Hypo direction accuracy: 93.9%  (original: 94.4%)
